# Практическая работа № 1. Titanic

**Цель:** построить единый Pipeline для предсказания выживания пассажира
(`survived`: 0 — не выжил, 1 — выжил), подобрать гиперпараметры
и оценить качество на отложенной тестовой выборке.

Все ячейки оставлены **без запуска**: результатов обучения и готовых метрик
в ноутбуке нет. Выполняйте ячейки последовательно сверху вниз.

Код оформлен по PEP 8: отступы в четыре пробела, понятные имена,
короткие строки, docstring и поясняющие комментарии.

## 1. Зависимости и импорты

Нужны Python 3.10+ и пакеты `pandas`, `scikit-learn`, `jupyterlab`.
Установите их в используемое окружение командой в терминале:

```bash
python -m pip install pandas scikit-learn jupyterlab
```

При чтении датасета по URL понадобится интернет. Вместо URL можно указать
путь к локальному CSV из того же источника.

In [ ]:
from pprint import pprint

import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted


RANDOM_STATE = 42
TEST_SIZE = 0.2
MISSING_COUNT_COLUMN = "missing_count"

## 2. Загрузка данных и выбор признаков

Используем [Titanic из репозитория seaborn-data](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv).
CSV читается напрямую через pandas; устанавливать seaborn не требуется.

Признаки выбираются явно:

| Тип | Признаки |
| --- | --- |
| Числовые | `age` — возраст; `sibsp` — число братьев, сестёр и супругов; `parch` — число родителей и детей; `fare` — стоимость билета |
| Категориальные | `pclass` — класс билета; `sex` — пол; `embarked` — порт посадки; `deck` — палуба |

`pclass` обрабатывается как категория, хотя хранится числом.
Столбец `alive` исключён: он дублирует целевую переменную и привёл бы
к утечке ответа. Остальные производные и дублирующие поля также не используем.
Пропуски на этом этапе сохраняем.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "mwaskom/seaborn-data/master/titanic.csv"
)

NUMERIC_COLUMNS = ["age", "sibsp", "parch", "fare"]
CATEGORICAL_COLUMNS = ["pclass", "sex", "embarked", "deck"]
FEATURE_COLUMNS = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS

data = pd.read_csv(DATA_URL)
x = data.loc[:, FEATURE_COLUMNS].copy()
y = data["survived"].astype(int)

print(f"Количество объектов: {len(x)}")
print(f"Количество исходных признаков: {x.shape[1]}")

## 3. Разделение на train и test

80% данных используем для обучения, 20% — для итоговой проверки.
`stratify=y` сохраняет близкие доли классов в обеих частях.
Фиксированный `random_state` позволяет воспроизвести разбиение.

Разделение выполняется **до обучения преобразований**. Медианы, самые частые
категории, параметры масштабирования и словарь OneHotEncoder будут вычисляться
внутри Pipeline только по обучающей части каждого разбиения кросс-валидации.
Тестовые данные не участвуют в подборе параметров.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train: {x_train.shape[0]} объектов")
print(f"Test: {x_test.shape[0]} объектов")
print("Пропуски в обучающей выборке:")
display(x_train.isna().sum().rename("missing_values").to_frame())

## 4. Кастомный трансформер количества пропусков

`MissingCountTransformer` добавляет числовой признак `missing_count` —
число пропусков среди **восьми выбранных исходных признаков** пассажира.
Целевая переменная в подсчёт не входит.

Трансформер стоит первым в Pipeline, поэтому считает пропуски
**до их заполнения**. Если поставить его после imputer, информация будет потеряна.
Исходный DataFrame не изменяется: возвращается его копия с новым столбцом.

Наследование от `BaseEstimator` и `TransformerMixin` обеспечивает совместимость
с Pipeline и клонированием в GridSearchCV. Метод `fit` запоминает схему колонок,
а `transform` проверяет её и вычисляет признак независимо для каждой строки.

In [ ]:
class MissingCountTransformer(TransformerMixin, BaseEstimator):
    """Добавить количество пропусков среди входных признаков строки."""

    def __init__(self, column_name="missing_count"):
        self.column_name = column_name

    def fit(self, x, y=None):
        """Запомнить колонки входного DataFrame и вернуть трансформер."""
        if not isinstance(x, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame.")
        if self.column_name in x.columns:
            raise ValueError("Колонка для числа пропусков уже существует.")

        self.feature_names_in_ = x.columns.to_numpy(copy=True)
        self.n_features_in_ = x.shape[1]
        return self

    def transform(self, x):
        """Вернуть копию DataFrame с числом пропусков в каждой строке."""
        check_is_fitted(self, "feature_names_in_")
        if not isinstance(x, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame.")
        if list(x.columns) != list(self.feature_names_in_):
            raise ValueError("Колонки и их порядок должны совпадать с fit.")

        transformed = x.copy()
        transformed[self.column_name] = x.isna().sum(axis=1)
        return transformed

## 5. Предобработка и общий Pipeline

Числовая ветка: `SimpleImputer(strategy="median")` →
`StandardScaler()`. Новый признак `missing_count` включён в эту же ветку.

Категориальная ветка: `SimpleImputer(strategy="most_frequent")` →
`OneHotEncoder(handle_unknown="ignore")`. Неизвестная при обучении категория
при предсказании не вызовет ошибку.

`ColumnTransformer` объединяет обе ветки. Общий Pipeline выполняет:
добавление количества пропусков → предобработку → `LogisticRegression`.
Увеличенный `max_iter` даёт оптимизатору больше итераций для сходимости.

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            NUMERIC_COLUMNS + [MISSING_COUNT_COLUMN],
        ),
        ("cat", categorical_pipeline, CATEGORICAL_COLUMNS),
    ],
    remainder="drop",
)

pipeline = Pipeline(
    steps=[
        (
            "missing_count",
            MissingCountTransformer(column_name=MISSING_COUNT_COLUMN),
        ),
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

## 6. Настройка GridSearchCV

Подбираем два гиперпараметра:

- `classifier__C`: обратная сила регуляризации. Меньшее значение означает
  более сильную регуляризацию.
- `classifier__class_weight`: `None` задаёт одинаковые веса объектов,
  `"balanced"` учитывает частоты классов обучающей части.

Медианное заполнение числовых пропусков остаётся фиксированным, чтобы
сохранить требование задания. Если отдельно исследовать стратегию заполнения,
правильное имя вложенного параметра — `preprocessor__num__imputer__strategy`
(между именами шагов и параметром используются **двойные подчёркивания**).

Проверяем 5 × 2 = 10 комбинаций на пяти стратифицированных разбиениях:
50 обучений при кросс-валидации и ещё одно на всём train для лучшего варианта.
Критерий выбора — средний ROC AUC. Он оценивает ранжирование пассажиров
по вероятности выживания без привязки к одному порогу.

`refit=True` переобучит лучший Pipeline на всём train.
`n_jobs=-1` разрешает использовать все доступные процессорные ядра.

In [ ]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "classifier__class_weight": [None, "balanced"],
}

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cross_validation,
    n_jobs=-1,
    refit=True,
    error_score="raise",
)

## 7. Обучение и лучшие параметры

**Следующая ячейка запускает обучение, когда вы выполните её самостоятельно.**
До неё создаются только данные и объекты; модель ещё не обучена.

В поиск передаются только `x_train` и `y_train`.
После завершения выводятся `best_params_` и лучший средний ROC AUC
кросс-валидации. Это оценка на разбиениях train, а не на тестовой выборке.

In [ ]:
grid_search.fit(x_train, y_train)

print("best_params:")
pprint(grid_search.best_params_)
print(f"Лучший ROC AUC на CV: {grid_search.best_score_:.4f}")

## 8. Итоговая оценка на тестовой выборке

Используем уже обученный `best_estimator_`; дополнительный `fit` не нужен.
Через него тестовые данные проходят те же преобразования, что и train.

- **Accuracy** — доля правильных предсказаний.
- **F1** — гармоническое среднее precision и recall для класса «выжил».
- **ROC AUC** — качество ранжирования, вычисляется по вероятностям класса 1.
- **Classification report** — precision, recall, F1 и число объектов
  отдельно для каждого класса.
- **Матрица ошибок** — строки соответствуют истинным классам,
  столбцы — предсказанным.

Тестовую оценку используем как итоговую проверку, а не как основание
для повторного подбора гиперпараметров на той же тестовой выборке.

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)
y_probability = best_model.predict_proba(x_test)[:, 1]

test_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_probability),
    },
    name="test_score",
)

display(test_metrics.to_frame().round(4))

print("Отчёт по классам:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Не выжил", "Выжил"],
        digits=4,
        zero_division=0,
    )
)

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Факт: не выжил", "Факт: выжил"],
    columns=["Прогноз: не выжил", "Прогноз: выжил"],
)
display(confusion)

## 9. Как сформулировать вывод после запуска

Укажите выведенные `best_params`, средний ROC AUC на CV
и значения Accuracy, F1, ROC AUC на test. По матрице ошибок объясните,
какие ошибки допускает модель. Сравнение CV и test помогает заметить
возможную нестабильность оценки, но само по себе не доказывает переобучение.

Численные результаты здесь намеренно не приведены: обучение не запускалось.

### Источники

- [Датасет Titanic](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv).
- [Документация GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).
- [Документация SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html).